In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)


# STEP 1: LOAD DATA

df = pd.read_csv('lab_sensor.csv')


# STEP 2: FEATURE ENGINEERING

# Create features

df['Delta'] = df['reading'].diff().fillna(0)

df['Rolling_Var'] = (
    df['reading']
    .rolling(window=5)
    .var()
    .fillna(0)
)


# STEP 3: CREATE CLASS LABELS

mean = df['reading'].mean()
std = df['reading'].std()

conditions = [
    (df['reading'] <= mean + std),

    (df['reading'] > mean + std) &
    (df['reading'] <= mean + 2 * std),

    (df['reading'] > mean + 2 * std)
]

df['Class_Label'] = np.select(
    conditions,
    [0, 1, 2],
    default=0
)


# STEP 4: FEATURE & TARGET

X = df[['reading', 'Delta', 'Rolling_Var']]

y = df['Class_Label']


# STEP 5: TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# STEP 6: MODEL TRAINING

clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42
)

clf.fit(X_train, y_train)


# STEP 7: PREDICTION

y_pred = clf.predict(X_test)


# STEP 8: EVALUATION

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)


Accuracy:
0.9

Confusion Matrix:
[[8 1]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94         9
           1       0.50      1.00      0.67         1

    accuracy                           0.90        10
   macro avg       0.75      0.94      0.80        10
weighted avg       0.95      0.90      0.91        10

